# Finale Präsentationsgrafiken und Story

Dieses Notebook bündelt die Grafiken für die Abschlusspräsentation. Es liest das finale Team-Match-Dataset und formt daraus eine zusammenhängende Ergebnisgeschichte: Wetter ist als Kontext sichtbar, aber Teamstärke erklärt Team-xG deutlich besser.

## Leitfrage

Wie hängen Wetterbedingungen und Teamstärke mit Spielstil und Chancenqualität bei internationalen Fußballturnieren zusammen?

Die Antwort entsteht nicht aus einem einzelnen Plot. Wir gehen Schritt für Schritt vor: erst Datenlogik und Scope, dann Wetterkontext, danach Spielstil und Chancen, und am Ende der Modellvergleich.

In [ ]:
from pathlib import Path
import math

import pandas as pd
from PIL import Image, ImageDraw, ImageFont

from project_paths import project_root

ROOT = project_root()
DATA_PATH = ROOT / 'data' / 'gold' / 'team_match_analysis_dataset.csv'
MODEL_COMPARISON_PATH = ROOT / 'outputs' / 'tables' / 'model_comparison.csv'
FIGURE_DIR = ROOT / 'outputs' / 'figures'
TABLE_DIR = ROOT / 'outputs' / 'tables'

FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

PALETTE = {
    'ink': '#222222',
    'muted': '#555f73',
    'grid': '#d6d6d6',
    'axis': '#c9c9c9',
    'bg': '#ffffff',
    'panel': '#f5f5f5',
    'blue': '#6679a8',
    'orange': '#c98d66',
    'green': '#6ca176',
    'teal': '#5c9a93',
    'red': '#b25f64',
    'purple': '#8b7caf',
    'pink': '#c791bb',
    'gray': '#8f8f8f',
    'olive': '#c0b783',
    'cyan': '#7fb0c0',
    'gold': '#d9a332',
    'outlier': '#6f7785',
}
COLOR_CYCLE = ['blue', 'orange', 'green', 'red', 'purple', 'pink', 'gray', 'olive', 'cyan']


def _font(size=28, bold=False):
    candidates = [
        'C:/Windows/Fonts/arialbd.ttf' if bold else 'C:/Windows/Fonts/arial.ttf',
        'C:/Windows/Fonts/segoeuib.ttf' if bold else 'C:/Windows/Fonts/segoeui.ttf',
    ]
    for candidate in candidates:
        if Path(candidate).exists():
            return ImageFont.truetype(candidate, size)
    return ImageFont.load_default()

FONT_TITLE = _font(46, True)
FONT_SUBTITLE = _font(26)
FONT_LABEL = _font(24)
FONT_SMALL = _font(20)
FONT_TINY = _font(17)
FONT_NUMBER = _font(46, True)


def _hex(color):
    color = color.lstrip('#')
    return tuple(int(color[i:i+2], 16) for i in (0, 2, 4))


def _text(draw, xy, text, fill='ink', font=None, anchor=None):
    draw.text(xy, str(text), fill=_hex(PALETTE.get(fill, fill)), font=font or FONT_LABEL, anchor=anchor)



def _canvas(title, subtitle=None, size=(1400, 820)):
    image = Image.new('RGB', size, _hex(PALETTE['bg']))
    draw = ImageDraw.Draw(image)
    _text(draw, (55, 36), title, font=FONT_TITLE)
    if subtitle:
        _text(draw, (55, 106), subtitle, fill='muted', font=FONT_SUBTITLE)
    return image, draw


def _save(image, path):
    image.save(path, quality=95)
    return path


def _nice_max(value):
    if value <= 0:
        return 1
    exp = math.floor(math.log10(value))
    base = 10 ** exp
    for mult in [1, 2, 5, 10]:
        if value <= mult * base:
            return mult * base
    return 10 * base


def _draw_plot_frame(draw, left, top, right, bottom):
    draw.rectangle([left, top, right, bottom], outline=_hex(PALETTE['axis']), width=1)


def draw_histogram(series, path, title, subtitle, xlabel, bins=14, color='blue'):
    values = pd.Series(series).dropna().astype(float)
    image, draw = _canvas(title, subtitle)
    left, top, right, bottom = 170, 185, 1325, 730
    min_v, max_v = float(values.min()), float(values.max())
    width = (max_v - min_v) / bins
    counts = []
    for i in range(bins):
        lo = min_v + i * width
        hi = min_v + (i + 1) * width
        if i == bins - 1:
            count = int(((values >= lo) & (values <= hi)).sum())
        else:
            count = int(((values >= lo) & (values < hi)).sum())
        counts.append(count)
    y_max = _nice_max(max(counts))
    _draw_plot_frame(draw, left, top, right, bottom)
    _text(draw, (left, top - 28), 'Anzahl Spiele', fill='muted', font=FONT_TINY)
    for tick in range(0, 6):
        y = bottom - (bottom - top) * tick / 5
        draw.line([(left, y), (right, y)], fill=_hex(PALETTE['grid']), width=1)
        _text(draw, (left - 16, y), f'{int(y_max * tick / 5)}', fill='muted', font=FONT_TINY, anchor='rm')
    bar_gap = 4
    bar_w = (right - left) / bins
    for i, count in enumerate(counts):
        x0 = left + i * bar_w + bar_gap
        x1 = left + (i + 1) * bar_w - bar_gap
        y0 = bottom - (bottom - top) * count / y_max
        draw.rectangle([x0, y0, x1, bottom], fill=_hex(PALETTE[color]))
    median = float(values.median())
    median_x = left + (median - min_v) / (max_v - min_v) * (right - left)
    draw.line([(median_x, top), (median_x, bottom)], fill=_hex(PALETTE['red']), width=3)
    _text(draw, (median_x + 8, top + 9), f'Median {median:.1f}', fill='red', font=FONT_TINY)
    for i in range(0, 6):
        x = left + (right - left) * i / 5
        value = min_v + (max_v - min_v) * i / 5
        _text(draw, (x, bottom + 22), f'{value:.0f}', fill='muted', font=FONT_TINY, anchor='mt')
    _text(draw, ((left + right) / 2, 810), xlabel, fill='ink', font=FONT_SMALL, anchor='mm')
    _text(draw, (left, 140), f'n={len(values)} | min={min_v:.1f} | max={max_v:.1f}', fill='muted', font=FONT_TINY)
    return _save(image, path)



def _box_stats(vals):
    q1, med, q3 = vals.quantile([0.25, 0.5, 0.75])
    iqr = q3 - q1
    lower_fence = q1 - 1.5 * iqr
    upper_fence = q3 + 1.5 * iqr
    non_outliers = vals[(vals >= lower_fence) & (vals <= upper_fence)]
    low = non_outliers.min() if len(non_outliers) else vals.min()
    high = non_outliers.max() if len(non_outliers) else vals.max()
    outliers = vals[(vals < low) | (vals > high)]
    return q1, med, q3, low, high, outliers


def draw_boxplot(groups, path, title, subtitle, ylabel, color='blue', value_fmt='{:.1f}', y_min=None, y_max=None):
    clean = [(label, pd.Series(vals).dropna().astype(float)) for label, vals in groups if len(pd.Series(vals).dropna())]
    all_values = pd.concat([vals for _, vals in clean])
    y_min = float(all_values.min()) if y_min is None else y_min
    y_max = float(all_values.max()) if y_max is None else y_max
    pad = (y_max - y_min) * 0.10 or 1
    y_min, y_max = y_min - pad, y_max + pad
    image, draw = _canvas(title, subtitle)
    left, top, right, bottom = 135, 170, 1325, 650
    _draw_plot_frame(draw, left, top, right, bottom)
    _text(draw, (left, top - 24), ylabel, fill='muted', font=FONT_TINY)
    for tick in range(0, 6):
        value = y_min + (y_max - y_min) * tick / 5
        y = bottom - (value - y_min) / (y_max - y_min) * (bottom - top)
        draw.line([(left, y), (right, y)], fill=_hex(PALETTE['grid']), width=1)
        _text(draw, (left - 14, y), value_fmt.format(value), fill='muted', font=FONT_TINY, anchor='rm')
    n = len(clean)
    step = (right - left) / n
    for idx, (label, vals) in enumerate(clean):
        q1, med, q3, low, high, outliers = _box_stats(vals)
        x = left + step * (idx + 0.5)
        box_w = min(110, step * 0.38)
        cap_w = box_w * 0.55
        fill_color = COLOR_CYCLE[idx % len(COLOR_CYCLE)] if color == 'cycle' else color
        def y(value):
            return bottom - (float(value) - y_min) / (y_max - y_min) * (bottom - top)
        draw.line([(x, y(low)), (x, y(high))], fill=_hex(PALETTE['ink']), width=2)
        draw.line([(x - cap_w / 2, y(low)), (x + cap_w / 2, y(low))], fill=_hex(PALETTE['ink']), width=2)
        draw.line([(x - cap_w / 2, y(high)), (x + cap_w / 2, y(high))], fill=_hex(PALETTE['ink']), width=2)
        draw.rectangle([x - box_w / 2, y(q3), x + box_w / 2, y(q1)], fill=_hex(PALETTE[fill_color]), outline=_hex(PALETTE['ink']), width=1)
        draw.line([(x - box_w / 2, y(med)), (x + box_w / 2, y(med))], fill=_hex(PALETTE['gold']), width=4)
        display_outliers = outliers.sort_values()
        if len(display_outliers) > 8:
            display_outliers = pd.concat([display_outliers.head(4), display_outliers.tail(4)])
        for j, value in enumerate(display_outliers):
            jitter = ((j % 5) - 2) * 2.0
            yy = y(value)
            draw.ellipse([x + jitter - 1.8, yy - 1.8, x + jitter + 1.8, yy + 1.8], fill=_hex(PALETTE['outlier']))
        _text(draw, (x, y(med) - 10), value_fmt.format(float(med)), fill='ink', font=FONT_TINY, anchor='mb')
        _text(draw, (x, bottom + 22), label, fill='ink', font=FONT_SMALL, anchor='mt')
        outlier_note = f'n={len(vals)}'
        if len(outliers):
            outlier_note += f' | Ausreißer={len(outliers)}'
        _text(draw, (x, bottom + 46), outlier_note, fill='muted', font=FONT_TINY, anchor='mt')
    return _save(image, path)

def draw_bar_chart(items, path, title, subtitle, ylabel='', color='blue', percent=False):
    image, draw = _canvas(title, subtitle)
    left, top, right, bottom = 170, 185, 1325, 730
    values = [float(item[1]) for item in items]
    y_max = _nice_max(max(values) * 1.2)
    _draw_plot_frame(draw, left, top, right, bottom)
    if ylabel:
        _text(draw, (left, top - 28), ylabel, fill='muted', font=FONT_TINY)
    for tick in range(0, 6):
        value = y_max * tick / 5
        y = bottom - (bottom - top) * tick / 5
        draw.line([(left, y), (right, y)], fill=_hex(PALETTE['grid']), width=1)
        if percent:
            tick_label = f'{value:.0f}%'
        elif y_max <= 1:
            tick_label = f'{value:.2f}'
        else:
            tick_label = f'{value:.0f}'
        _text(draw, (left - 16, y), tick_label, fill='muted', font=FONT_TINY, anchor='rm')
    step = (right - left) / len(items)
    bar_w = min(125, step * 0.5)
    for idx, (label, value) in enumerate(items):
        x = left + step * (idx + 0.5)
        y = bottom - (bottom - top) * float(value) / y_max
        fill_color = COLOR_CYCLE[idx % len(COLOR_CYCLE)] if color == 'cycle' else color
        draw.rectangle([x - bar_w / 2, y, x + bar_w / 2, bottom], fill=_hex(PALETTE[fill_color]))
        text_value = f'{value:.1f}%' if percent else f'{value:.3f}' if value < 1 else f'{value:.1f}'
        _text(draw, (x, y - 12), text_value, fill='ink', font=FONT_SMALL, anchor='mb')
        _text(draw, (x, bottom + 25), label, fill='ink', font=FONT_SMALL, anchor='mt')
    return _save(image, path)



def draw_two_panel_boxplot(left_groups, right_groups, path, title, subtitle):
    image, draw = _canvas(title, subtitle)
    panels = [
        (left_groups, (115, 200, 665, 630), 'Passquote nach Temperaturgruppe', 'Passquote', '{:.0f}%'),
        (right_groups, (805, 200, 1355, 630), 'Long-Ball-Share nach Regen', 'Long-Ball-Share', '{:.0f}%'),
    ]
    for panel_idx, (groups, area, panel_title, y_label, fmt) in enumerate(panels):
        left, top, right, bottom = area
        clean = [(label, pd.Series(vals).dropna().astype(float) * 100) for label, vals in groups if len(pd.Series(vals).dropna())]
        all_values = pd.concat([vals for _, vals in clean])
        y_min, y_max = float(all_values.min()), float(all_values.max())
        pad = (y_max - y_min) * 0.13 or 1
        y_min, y_max = y_min - pad, y_max + pad
        _text(draw, (left, top - 42), panel_title, font=FONT_LABEL)
        _text(draw, (left, top - 18), y_label, fill='muted', font=FONT_TINY)
        _draw_plot_frame(draw, left, top, right, bottom)
        for tick in range(0, 5):
            value = y_min + (y_max - y_min) * tick / 4
            y = bottom - (value - y_min) / (y_max - y_min) * (bottom - top)
            draw.line([(left, y), (right, y)], fill=_hex(PALETTE['grid']), width=1)
            _text(draw, (left - 12, y), fmt.format(value), fill='muted', font=FONT_TINY, anchor='rm')
        n = len(clean)
        step = (right - left) / n
        for idx, (label, vals) in enumerate(clean):
            q1, med, q3, low, high, outliers = _box_stats(vals)
            x = left + step * (idx + 0.5)
            box_w = min(95, step * 0.36)
            cap_w = box_w * 0.55
            fill_color = COLOR_CYCLE[(panel_idx * 3 + idx) % len(COLOR_CYCLE)]
            def y(value):
                return bottom - (float(value) - y_min) / (y_max - y_min) * (bottom - top)
            draw.line([(x, y(low)), (x, y(high))], fill=_hex(PALETTE['ink']), width=2)
            draw.line([(x - cap_w / 2, y(low)), (x + cap_w / 2, y(low))], fill=_hex(PALETTE['ink']), width=2)
            draw.line([(x - cap_w / 2, y(high)), (x + cap_w / 2, y(high))], fill=_hex(PALETTE['ink']), width=2)
            draw.rectangle([x - box_w / 2, y(q3), x + box_w / 2, y(q1)], fill=_hex(PALETTE[fill_color]), outline=_hex(PALETTE['ink']), width=1)
            draw.line([(x - box_w / 2, y(med)), (x + box_w / 2, y(med))], fill=_hex(PALETTE['gold']), width=4)
            display_outliers = outliers.sort_values()
            if len(display_outliers) > 8:
                display_outliers = pd.concat([display_outliers.head(4), display_outliers.tail(4)])
            for j, value in enumerate(display_outliers):
                jitter = ((j % 5) - 2) * 2.0
                yy = y(value)
                draw.ellipse([x + jitter - 1.8, yy - 1.8, x + jitter + 1.8, yy + 1.8], fill=_hex(PALETTE['outlier']))
            _text(draw, (x, y(med) - 10), fmt.format(float(med)), fill='ink', font=FONT_TINY, anchor='mb')
            _text(draw, (x, bottom + 22), label, fill='ink', font=FONT_SMALL, anchor='mt')
            outlier_note = f'n={len(vals)}'
            if len(outliers):
                outlier_note += f' | Ausreißer={len(outliers)}'
            _text(draw, (x, bottom + 46), outlier_note, fill='muted', font=FONT_TINY, anchor='mt')
    return _save(image, path)


def draw_flow(path):
    image, draw = _canvas('Analyse-Logik', 'Vom Rohsignal zur Präsentationsaussage: drei Quellen, eine Team-Match-Ebene, vier Analyseblöcke.')
    boxes = [
        ('StatsBomb\nEvents', 85, 190, 'blue'),
        ('Open-Meteo\nWetter', 530, 190, 'teal'),
        ('Elo\nRatings', 975, 190, 'gold'),
        ('Team-Match-\nDataset', 530, 385, 'panel'),
    ]
    for text, x, y, color in boxes:
        fill = PALETTE[color] if color != 'panel' else PALETTE['panel']
        outline = PALETTE['ink'] if color == 'panel' else fill
        draw.rounded_rectangle([x, y, x + 310, y + 105], radius=13, fill=_hex(fill), outline=_hex(outline), width=2)
        text_color = 'ink' if color == 'panel' else '#ffffff'
        _text(draw, (x + 155, y + 52), text, fill=text_color, font=FONT_SMALL, anchor='mm')
    for x0 in [240, 685, 1130]:
        draw.line([(x0, 305), (685, 385)], fill=_hex(PALETTE['muted']), width=3)
    draw.line([(685, 490), (685, 570)], fill=_hex(PALETTE['muted']), width=3)
    draw.polygon([(685, 570), (673, 550), (697, 550)], fill=_hex(PALETTE['muted']))
    chain = ['Wetterverteilung', 'Spielstil', 'Chancenqualität', 'Modellvergleich']
    x = 95
    for idx, label in enumerate(chain):
        draw.rounded_rectangle([x, 615, x + 245, 690], radius=10, fill=_hex('#eef2f6'), outline=_hex(PALETTE['axis']), width=1)
        _text(draw, (x + 122, 652), label, font=FONT_TINY, anchor='mm')
        if idx < len(chain) - 1:
            draw.line([(x + 252, 652), (x + 315, 652)], fill=_hex(PALETTE['muted']), width=3)
            draw.polygon([(x + 315, 652), (x + 297, 642), (x + 297, 662)], fill=_hex(PALETTE['muted']))
        x += 315
    return _save(image, path)


def draw_scope(team_df, weather_df, path):
    image, draw = _canvas('Datensatz & Scope', 'Die Analyse arbeitet auf Team-Match-Ebene: ein Spiel wird aus Sicht beider Teams betrachtet.')
    metrics = [
        ('Turniere', team_df['short_name'].nunique()),
        ('Matches', weather_df['match_id'].nunique()),
        ('Team-Match-Zeilen', len(team_df)),
        ('Teams', team_df['team_name'].nunique()),
    ]
    x_positions = [70, 395, 720, 1045]
    for (label, value), x in zip(metrics, x_positions):
        draw.rounded_rectangle([x, 170, x + 255, 310], radius=10, fill=_hex(PALETTE['panel']), outline=_hex(PALETTE['axis']), width=1)
        _text(draw, (x + 127, 220), f'{int(value):,}'.replace(',', '.'), font=FONT_NUMBER, anchor='mm')
        _text(draw, (x + 127, 270), label, fill='muted', font=FONT_SMALL, anchor='mm')
    tournament_counts = weather_df['short_name'].value_counts().sort_values(ascending=True)
    left, top, right, bottom = 235, 395, 1225, 705
    max_count = tournament_counts.max()
    step = (bottom - top) / len(tournament_counts)
    _text(draw, (235, 355), 'Matches je Turnier', font=FONT_SMALL)
    for idx, (label, count) in enumerate(tournament_counts.items()):
        y = top + step * idx + step * 0.5
        bar_len = (right - left) * count / max_count
        fill_color = COLOR_CYCLE[idx % len(COLOR_CYCLE)]
        draw.rounded_rectangle([left, y - 16, left + bar_len, y + 16], radius=4, fill=_hex(PALETTE[fill_color]))
        _text(draw, (left - 14, y), label, fill='ink', font=FONT_TINY, anchor='rm')
        _text(draw, (left + bar_len + 10, y), f'{count} Matches', fill='muted', font=FONT_TINY, anchor='lm')
    return _save(image, path)

PRESENTATION_FLOW_PATH = FIGURE_DIR / 'presentation_analysis_flow.png'
PRESENTATION_SCOPE_PATH = FIGURE_DIR / 'presentation_dataset_scope.png'
PRESENTATION_WEATHER_PATH = FIGURE_DIR / 'presentation_weather_distribution.png'
PRESENTATION_ELO_XG_PATH = FIGURE_DIR / 'presentation_xg_by_elo_group.png'
PRESENTATION_INTENSITY_PATH = FIGURE_DIR / 'presentation_pressures_by_temperature_group.png'
PRESENTATION_BALL_CONTROL_PATH = FIGURE_DIR / 'presentation_ball_control_weather.png'
PRESENTATION_CHANCE_PATH = FIGURE_DIR / 'presentation_xg_by_temperature_group.png'
PRESENTATION_MODEL_PATH = FIGURE_DIR / 'presentation_model_r2_comparison.png'

required_columns = [
    'match_id', 'short_name', 'team_name', 'temperature_c', 'feels_like_c', 'rain_mm', 'rain_flag',
    'weather_missing_flag', 'elo_group', 'xg', 'xg_per_shot', 'pressures',
    'pass_completion_rate', 'long_ball_share', 'shots', 'elo_diff',
]


## Datenbasis

Für reine Wetterverteilungen wird jedes Match nur einmal gezählt. Für Spielstil, Chancenqualität und Teamstärke bleibt die Team-Match-Ebene erhalten, weil jede Mannschaft im selben Spiel eigene Werte für xG, Pressures und Passquote hat.

In [ ]:
assert DATA_PATH.exists(), f'Missing required input file: {DATA_PATH}'
team_match_df = pd.read_csv(DATA_PATH)
missing_columns = [column for column in required_columns if column not in team_match_df.columns]
assert not missing_columns, f'Missing required columns: {missing_columns}'

weather_columns = ['match_id', 'short_name', 'temperature_c', 'feels_like_c', 'rain_mm', 'rain_flag', 'weather_missing_flag']
weather_df = (
    team_match_df[weather_columns]
    .drop_duplicates(subset=['match_id'])
    .sort_values(['short_name', 'match_id'])
    .reset_index(drop=True)
)

assert weather_df['match_id'].is_unique
assert not weather_df['weather_missing_flag'].any()

{
    'matches': int(weather_df['match_id'].nunique()),
    'team_match_rows': int(len(team_match_df)),
    'tournaments': int(weather_df['short_name'].nunique()),
    'teams': int(team_match_df['team_name'].nunique()),
}

## Von drei Quellen zur Analyseidee

Die erste Grafik ersetzt den technischen DataGraph in der Präsentation. Sie zeigt nicht jede Pipeline-Stufe, sondern die Denklogik der Analyse.

In [ ]:
draw_flow(PRESENTATION_FLOW_PATH)

![Analyse-Logik](../outputs/figures/presentation_analysis_flow.png)

StatsBomb beschreibt das Verhalten auf dem Platz, Open-Meteo den Wetterkontext am Spielort und Elo die relative Teamstärke. Erst auf Team-Match-Ebene werden diese Perspektiven vergleichbar: ein Spiel wird aus Sicht beider Teams gelesen.

So entsteht die Dramaturgie: Wetterverteilung, Spielstil, Chancenqualität und Modellvergleich bauen aufeinander auf.

## Was genau wird verglichen?

Bevor wir Effekte interpretieren, braucht die Präsentation ein Gefühl für den Scope.

In [ ]:
draw_scope(team_match_df, weather_df, PRESENTATION_SCOPE_PATH)

![Datensatz und Scope](../outputs/figures/presentation_dataset_scope.png)

Die sechs Turniere ergeben 314 Matches. Da wir pro Team und Match analysieren, entstehen 628 Team-Match-Zeilen. Diese Ebene ist wichtig, weil xG, Pressures, Passquote und Long-Ball-Share immer aus Sicht eines Teams gemessen werden.

Der Datensatz ist groß genug für explorative Vergleiche, aber klein genug, dass Ausreißer und Gruppengrößen sichtbar mitgedacht werden müssen.

## Wetter ist im Datensatz sichtbar

Jetzt prüfen wir, ob Wetter überhaupt genug Streuung hat, um als Kontextvariable interessant zu sein.

In [ ]:
tournament_order = list(weather_df['short_name'].drop_duplicates())
draw_boxplot(
    [(tournament, weather_df.loc[weather_df['short_name'] == tournament, 'temperature_c']) for tournament in tournament_order],
    PRESENTATION_WEATHER_PATH,
    'Wetter im Datensatz',
    f'Temperatur variiert deutlich; Regen bleibt eine kleine Gruppe ({int(weather_df["rain_flag"].sum())}/{len(weather_df)} Matches).',
    'Temperatur in °C',
    color='cycle',
)

![Wetter im Datensatz](../outputs/figures/presentation_weather_distribution.png)

Die Turniere decken unterschiedliche Temperaturbereiche ab. Einige Wettbewerbe liegen klar im warmen Bereich, andere haben eine breitere Streuung. Regen ist vorhanden, bleibt aber eine kleine Gruppe.

Für die Präsentation heißt das: Temperaturgruppen eignen sich besser für Vergleiche; Regen ist eher ein ergänzendes Kontextsignal.

## Teamstärke als sportlicher Vergleichsanker

Bevor Wetter interpretiert wird, brauchen wir einen sportlichen Referenzpunkt. Elo liefert genau diesen Vergleichsanker.

In [ ]:
elo_order = ['underdog', 'balanced', 'favorite']
draw_boxplot(
    [(group, team_match_df.loc[team_match_df['elo_group'] == group, 'xg']) for group in elo_order],
    PRESENTATION_ELO_XG_PATH,
    'Teamstärke als Vergleichsanker',
    'Favoriten erzeugen im Median mehr xG als Außenseiter.',
    'Team-xG',
    color='cycle',
)

![xG nach Elo-Gruppe](../outputs/figures/presentation_xg_by_elo_group.png)

Favoriten erzeugen im Median mehr xG als Außenseiter. Das passt zur sportlichen Erwartung: stärkere Teams kommen häufiger in gute Abschlusspositionen.

Diese Beobachtung ist wichtig für die spätere Punchline. Wenn Elo bereits sichtbar mit xG zusammenhängt, dürfen Wetterplots nicht isoliert überinterpretiert werden.

## Intensität: sichtbare Streuung statt einfacher Effekt

Pressures sind unsere wichtigste Intensitätsmetrik. Der Boxplot ist hier hilfreicher als eine Punktwolke, weil Median, Streuung und Ausreißer direkt sichtbar werden.

In [ ]:
temperature_order = ['mild', 'warm', 'hot']
draw_boxplot(
    [(group, team_match_df.loc[team_match_df['temperature_group'] == group, 'pressures']) for group in temperature_order],
    PRESENTATION_INTENSITY_PATH,
    'Wetter & Intensität',
    'Pressures nach Temperaturgruppe: Boxplots zeigen Median und Streuung besser als eine Punktwolke.',
    'Pressures pro Team-Match',
    color='cycle',
)

![Pressures nach Temperaturgruppe](../outputs/figures/presentation_pressures_by_temperature_group.png)

Die Medianwerte unterscheiden sich nicht dramatisch, gleichzeitig gibt es einzelne sehr intensive Team-Matches. Die Ausreißer werden bewusst nur dezent gezeigt, weil sie erklären, warum einzelne Extremspiele die Interpretation verzerren können.

Die saubere Aussage lautet: Temperaturgruppen zeigen Unterschiede in der Verteilung, aber kein eindeutiges Signal, das allein auf Wetter zurückgeführt werden sollte.

## Ballkontrolle: Präzision und Spielanlage

Ballkontrolle betrachten wir über zwei Perspektiven: Passquote als Präzision und Long-Ball-Share als Hinweis auf Spielanlage.

In [ ]:
draw_two_panel_boxplot(
    [(group, team_match_df.loc[team_match_df['temperature_group'] == group, 'pass_completion_rate']) for group in temperature_order],
    [('trocken', team_match_df.loc[~team_match_df['rain_flag'], 'long_ball_share']), ('Regen', team_match_df.loc[team_match_df['rain_flag'], 'long_ball_share'])],
    PRESENTATION_BALL_CONTROL_PATH,
    'Wetter & Ballkontrolle',
    'Passquote und Long-Ball-Anteil zeigen, ob Wetter eher Präzision oder Spielanlage berührt.',
)

![Wetter und Ballkontrolle](../outputs/figures/presentation_ball_control_weather.png)

Die Passquote bleibt über Temperaturgruppen hinweg relativ stabil. Beim Long-Ball-Share ist der Regenvergleich interessant, aber die Regen-Gruppe ist deutlich kleiner.

Damit wird die Aussage differenziert: Wetter kann als Kontext sichtbar sein, aber die Verteilungen sprechen nicht für einen einfachen, linearen Wettereinfluss auf Ballkontrolle.

## Chancenqualität bleibt vorsichtig zu lesen

Team-xG fasst Chancenmenge und Chancenqualität zusammen. Genau deshalb ist die Streuung hier besonders wichtig.

In [ ]:
draw_boxplot(
    [(group, team_match_df.loc[team_match_df['temperature_group'] == group, 'xg']) for group in temperature_order],
    PRESENTATION_CHANCE_PATH,
    'Wetter & Chancenqualität',
    'Team-xG unterscheidet sich zwischen Temperaturgruppen nur vorsichtig interpretierbar.',
    'Team-xG',
    color='cycle',
)

![xG nach Temperaturgruppe](../outputs/figures/presentation_xg_by_temperature_group.png)

Die Temperaturgruppen unterscheiden sich leicht, aber die Streuung ist groß. Einzelne Team-Matches erzeugen sehr hohe xG-Werte und werden als Ausreißer markiert.

Für die Präsentation ist das eine vorsichtige Folie: Wettergruppen schwanken, aber Wetter allein erklärt Team-xG nicht überzeugend.

## Der Modellvergleich setzt die Gewichtung

Am Ende vergleichen wir Wetter und Teamstärke direkt als Erklärungsfaktoren für Team-xG.

In [ ]:
assert MODEL_COMPARISON_PATH.exists(), f'Missing model comparison table: {MODEL_COMPARISON_PATH}'
model_df = pd.read_csv(MODEL_COMPARISON_PATH)
shown_models = ['weather_only', 'elo_only']
target_order = ['team_xg', 'successful_passes']
model_labels = {'weather_only': 'Wetter', 'elo_only': 'Elo'}
target_labels = {'team_xg': 'Team-xG', 'successful_passes': 'Erfolgreiche Pässe'}

plot_df = model_df.loc[
    model_df['target'].isin(target_order)
    & model_df['model_name'].isin(shown_models)
].copy()
assert len(plot_df) == 4, 'Expected weather and Elo models for both presentation targets.'

image, draw = _canvas(
    'Modellvergleich: Was erklärt mehr?',
    'R² für Team-xG und erfolgreiche Pässe.'
)
left, top, right, bottom = 115, 205, 1320, 705
panel_gap = 90
panel_w = (right - left - panel_gap) / 2
max_value = max(float(plot_df['r2'].max()), 0.01) * 1.25
colors = ['blue', 'gold']

for panel_idx, target in enumerate(target_order):
    panel_left = left + panel_idx * (panel_w + panel_gap)
    panel_right = panel_left + panel_w
    _text(draw, (panel_left, top - 50), target_labels[target], font=FONT_LABEL)
    _draw_plot_frame(draw, panel_left, top, panel_right, bottom)
    for tick in range(0, 5):
        y = bottom - (bottom - top) * tick / 4
        value = max_value * tick / 4
        draw.line([(panel_left, y), (panel_right, y)], fill=_hex(PALETTE['grid']), width=1)
        if panel_idx == 0:
            _text(draw, (panel_left - 16, y), f'{value:.2f}', fill='muted', font=FONT_TINY, anchor='rm')
    panel_model_df = plot_df.set_index(['target', 'model_name']).loc[target].loc[shown_models].reset_index()
    bar_w = 105
    x_positions = [panel_left + panel_w * 0.34, panel_left + panel_w * 0.66]
    for x, (_, row), color_key, model_name in zip(x_positions, panel_model_df.iterrows(), colors, shown_models):
        value = float(row['r2'])
        y0 = bottom - (bottom - top) * value / max_value
        draw.rounded_rectangle([x - bar_w / 2, y0, x + bar_w / 2, bottom], radius=9, fill=_hex(PALETTE[color_key]))
        _text(draw, (x, y0 - 15), f'{value:.3f}', font=FONT_SMALL, anchor='mm')
        _text(draw, (x, bottom + 33), model_labels[model_name], fill='muted', font=FONT_SMALL, anchor='mm')

_text(draw, (left, 755), 'Lesart: Höhere Balken bedeuten mehr erklärte Varianz. Für beide Zielgrößen liegt Elo klar vor Wetter.', fill='muted', font=FONT_SMALL)
_save(image, PRESENTATION_MODEL_PATH)
plot_df[['target', 'model_name', 'r2', 'adjusted_r2', 'rmse']]


![Modellvergleich R²](../outputs/figures/presentation_model_r2_comparison.png)

Der Unterschied ist in beiden Zielgrößen sichtbar: Wettervariablen allein erklären nur sehr wenig Varianz. Elo erklärt sowohl Team-xG als auch erfolgreiche Pässe deutlich besser.

Damit wird die zentrale Aussage rund: Wetter ist ein sichtbarer Kontextfaktor, aber Teamstärke ist der stärkere Erklärungsfaktor.


## Fazit

Wetter ist im Datensatz sichtbar: Temperatur unterscheidet sich zwischen Turnieren, Regen kommt vor, bleibt aber selten.

Spielstil- und Chancenmetriken schwanken zwischen Wettergruppen. Die Boxplots zeigen aber auch, dass Streuung und Ausreißer groß genug sind, um vorsichtig zu bleiben.

Die stärkste Ergebnisbotschaft kommt aus dem Modellvergleich: Elo beziehungsweise Teamstärke erklärt Team-xG deutlich besser als Wetter. Wetter ist Kontext, nicht Haupttreiber.

In [ ]:
presentation_paths = [
    PRESENTATION_FLOW_PATH,
    PRESENTATION_SCOPE_PATH,
    PRESENTATION_WEATHER_PATH,
    PRESENTATION_ELO_XG_PATH,
    PRESENTATION_INTENSITY_PATH,
    PRESENTATION_BALL_CONTROL_PATH,
    PRESENTATION_CHANCE_PATH,
    PRESENTATION_MODEL_PATH,
]

for path in presentation_paths:
    assert path.exists(), f'Missing expected figure: {path}'
    assert path.stat().st_size > 1_000, f'Figure looks too small or empty: {path}'

pd.DataFrame({
    'figure': [str(path.relative_to(ROOT)) for path in presentation_paths],
    'bytes': [path.stat().st_size for path in presentation_paths],
})